# Notebook 0 - Setup and EDA

Run this notebook to validate the dataset layout, inspect the cleaned labels,
and create the shared train/validation/test split files used by the experiment
notebooks.

Optional download steps stay separate from the default workflow because they
create many files and require Kaggle credentials.

In [ ]:
#@title 1. Load Project Configuration
from pathlib import Path
import sys

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/wild_animal_image_classification'),
    Path('/content'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'configs' / 'config.py').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from configs.config import CFG

CFG.ensure_output_dirs()
PROJECT_ROOT = CFG.runtime_root
DATA_DIR = CFG.raw_data_dir
TRAIN_IMAGES_DIR = CFG.train_images_dir
TRAIN_JSON = CFG.train_json
RESULTS_DIR = CFG.results_dir
SPLITS_DIR = CFG.splits_dir

print(f'Runtime root: {PROJECT_ROOT}')
print(f'Raw data dir: {DATA_DIR}')
print(f'Splits dir: {SPLITS_DIR}')

In [ ]:
#@title 2. Dependency Note
print('Dependencies are managed in the project environment. This notebook does not install packages.')

In [ ]:
#@title 3. Optional Manual Download Step
print('Optional step: place Kaggle credentials outside the notebook and download the raw dataset manually if data/raw is still empty.')

In [ ]:
#@title 4. Verify Raw Data Layout
from pathlib import Path

raw_dir = Path(DATA_DIR)
required_paths = [
    Path(TRAIN_IMAGES_DIR),
    Path(TRAIN_JSON),
]
for required_path in required_paths:
    print(f'{required_path}: {required_path.exists()}')

if raw_dir.exists():
    entries = sorted(path.name for path in raw_dir.iterdir())
    print('Raw data entries:', entries[:10])
else:
    print('Raw data directory does not exist yet.')

In [ ]:
#@title 5. Import Shared Utilities
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils.dataset import get_category_names, load_annotations, print_class_distribution, split_dataset

print('Shared utilities loaded successfully.')

In [ ]:
#@title 6. Load Annotations & Remove Empty Images
import json
import pandas as pd
import numpy as np
from collections import Counter

# --- Find the annotation JSON ---
DATA_DIR = f'{PROJECT_ROOT}/data'
possible_jsons = [
    f'{DATA_DIR}/iwildcam2019_train_annotations.json',
    f'{DATA_DIR}/train_annotations.json',
    f'{DATA_DIR}/iwildcam2019_train.json',
]
TRAIN_JSON = None
for p in possible_jsons:
    if os.path.exists(p):
        TRAIN_JSON = p
        break
if TRAIN_JSON is None:
    # Search for it
    import glob
    found = glob.glob(f'{DATA_DIR}/**/*train*.json', recursive=True)
    print('Found JSON files:', found)
    TRAIN_JSON = found[0] if found else None

print(f'Using annotations: {TRAIN_JSON}')

# --- Find images directory ---
possible_img_dirs = [
    f'{DATA_DIR}/train_images',
    f'{DATA_DIR}/train',
]
IMG_DIR = None
for p in possible_img_dirs:
    if os.path.isdir(p):
        IMG_DIR = p
        break
print(f'Using images dir: {IMG_DIR}')

In [ ]:
#@title 7. Parse Annotations into DataFrame
with open(TRAIN_JSON, 'r') as f:
    data = json.load(f)

print(f'Keys in JSON: {list(data.keys())}')
print(f'Number of images: {len(data["images"])}')
print(f'Number of annotations: {len(data["annotations"])}')
print(f'Number of categories: {len(data.get("categories", []))}')

# Show categories
if 'categories' in data:
    for cat in data['categories']:
        print(f"  ID {cat['id']:>3}: {cat['name']}")

In [ ]:
#@title 8. Build DataFrame & Remove Empty
images_lookup = {img['id']: img for img in data['images']}
ann_lookup = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in ann_lookup:
        ann_lookup[img_id] = ann['category_id']

rows = []
for img_id, img_info in images_lookup.items():
    cat_id = ann_lookup.get(img_id, 0)
    fname = img_info['file_name']
    rows.append({
        'image_id': img_id,
        'file_name': fname,
        'full_path': os.path.join(IMG_DIR, fname),
        'category_id': cat_id,
        'location': img_info.get('location', -1),
    })

df_all = pd.DataFrame(rows)
print(f'Total images: {len(df_all)}')
print(f'Empty images (category_id=0): {(df_all["category_id"]==0).sum()}')

# Remove empty
df = df_all[df_all['category_id'] != 0].reset_index(drop=True)
print(f'After removing empty: {len(df)} images')

# Remap to contiguous labels
unique_cats = sorted(df['category_id'].unique())
cat_to_label = {cat: i for i, cat in enumerate(unique_cats)}
label_to_cat = {i: cat for cat, i in cat_to_label.items()}
df['label'] = df['category_id'].map(cat_to_label)

NUM_CLASSES = len(unique_cats)
print(f'Number of animal classes: {NUM_CLASSES}')

In [ ]:
#@title 9. EDA — Class Distribution
import matplotlib.pyplot as plt
import seaborn as sns

cat_names = {c['id']: c['name'] for c in data.get('categories', [])}

counts = df['category_id'].value_counts().sort_index()
labels_for_plot = [cat_names.get(cid, f'cat_{cid}') for cid in counts.index]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(counts)), counts.values, color='steelblue')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(labels_for_plot, rotation=45, ha='right')
ax.set_ylabel('Number of Images')
ax.set_title('iWildCam 2019 — Class Distribution (Empty Removed)')
ax.grid(axis='y', alpha=0.3)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/class_distribution.png', dpi=150)
plt.show()
print('Saved class distribution plot.')

In [ ]:
#@title 10. EDA — Sample Images
from PIL import Image
import random

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
sample_labels = random.sample(range(NUM_CLASSES), min(15, NUM_CLASSES))

for i, ax in enumerate(axes.flat):
    if i < len(sample_labels):
        label = sample_labels[i]
        cat_id = label_to_cat[label]
        sample_row = df[df['label'] == label].sample(1).iloc[0]
        try:
            img = Image.open(sample_row['full_path']).convert('RGB')
            ax.imshow(img)
            name = cat_names.get(cat_id, f'cat_{cat_id}')
            ax.set_title(name, fontsize=10)
        except Exception as e:
            ax.set_title(f'Error: {e}', fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Images Per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/sample_images.png', dpi=150)
plt.show()

In [ ]:
#@title 11. EDA — Images per Location
loc_counts = df['location'].value_counts()
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(loc_counts)), loc_counts.values, color='coral', width=1.0)
ax.set_xlabel('Location (sorted by count)')
ax.set_ylabel('Number of Images')
ax.set_title(f'Images per Camera Location ({len(loc_counts)} locations)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/images_per_location.png', dpi=150)
plt.show()

In [ ]:
#@title 12. Create Stratified Splits & Save
from sklearn.model_selection import train_test_split

VAL_SIZE = 0.15
TEST_SIZE = 0.15
SEED = 42

train_val_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df['label'], random_state=SEED
)
relative_val = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df, test_size=relative_val, stratify=train_val_df['label'],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Save splits so all notebooks use the SAME data
SPLITS_DIR = CFG.splits_dir
os.makedirs(SPLITS_DIR, exist_ok=True)

train_df.to_csv(f'{SPLITS_DIR}/train.csv', index=False)
val_df.to_csv(f'{SPLITS_DIR}/val.csv', index=False)
test_df.to_csv(f'{SPLITS_DIR}/test.csv', index=False)

# Save mappings
import json
with open(f'{SPLITS_DIR}/cat_to_label.json', 'w') as f:
    json.dump({str(k): v for k, v in cat_to_label.items()}, f)
with open(f'{SPLITS_DIR}/label_to_cat.json', 'w') as f:
    json.dump({str(k): v for k, v in label_to_cat.items()}, f)
with open(f'{SPLITS_DIR}/cat_names.json', 'w') as f:
    json.dump({str(k): v for k, v in cat_names.items()}, f)

metadata = {
    'num_classes': NUM_CLASSES,
    'total_images': len(df),
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'seed': SEED,
}
with open(f'{SPLITS_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\nSplits saved to {SPLITS_DIR}/')
print('All team members should use these exact splits.')

In [ ]:
#@title 13. Verify a Few Images Load Correctly
from PIL import Image

errors = 0
for idx in range(min(100, len(train_df))):
    try:
        img = Image.open(train_df.iloc[idx]['full_path']).convert('RGB')
    except Exception as e:
        errors += 1
        print(f'Error loading {train_df.iloc[idx]["full_path"]}: {e}')

print(f'\nChecked 100 images — {errors} errors found.')
if errors == 0:
    print('All good! Data is ready for training.')